In [1]:
import pandas as pd

In [ ]:
gata2_phenotypes = pd.read_excel(r"GATA2_Phenotypes_Final_v8.xlsx")
gata2_phenotypes

,Phenotype_Index,Phenotype,Category,Category_Rank,New_Category,Order_Phenotype,Phenotype_Description_Expanded,Overlapping Phenotype Codes,Phenotype_Description,Phenotype_Description_v2,...,Rank,ICD10_code,Present_in_GATA2,Present_in_UKB,Keep_NonZero,Keep_WithZero,Keep_Adults,Keep_OG,Keep_Everything,Keep_Combinations
0,P100,head_neck_eyes_ptosis,Head Neck Eyes,1.0,Head and neck,1.01,Ptosis,NaN,Ptosis,Ptosis,...,49,"['H024','Q100']",Yes,Yes,Yes,Yes,Yes,Yes,Yes,Yes
1,P102,head_neck_eyes_epicanthic_folds,Head Neck Eyes,1.0,Head and neck,1.02,Epicanthic folds,NaN,Epicanthic folds,Epicanthic folds,...,51,['Q103'],Yes,Yes,Yes,Yes,Yes,Yes,Yes,Yes
2,P101,head_neck_eyes_hypertelorism,Head Neck Eyes,1.0,Head and neck,1.03,Hypertelorism,NaN,Hypertelorism,Hypertelorism,...,53,['Q752'],Yes,Yes,Yes,Yes,Yes,Yes,Yes,Yes
3,P105,head_neck_eyes_strabismus,Head Neck Eyes,1.0,Head and neck,1.04,Strabismus,NaN,Strabismus,Strabismus,...,55,"['H49','H500', 'H509', 'H501', 'H502', 'H506',...",Yes,Yes,Yes,Yes,No,Yes,Yes,Yes
4,P104,head_neck_eyes_small_palpebral_fissures,Head Neck Eyes,1.0,Head and neck,1.05,Short palpebral fissures,NaN,Short palpebral fissures,Short palpebral fissures,...,54,['H025'],Yes,Yes,Yes,Yes,Yes,Yes,Yes,Yes
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
124,P12,head_neck_eyes_treatment_associated_hearing_loss,Head Neck Eyes,NaN,Head and neck,Sensorineural,Treatment-associated hearing loss (Combined to...,NaN,Treatment-associated hearing loss (Combined to...,Treatment-associated hearing loss (Combined to...,...,38,"['H900', 'H901', 'H902', 'H903', 'H904', 'H906...",Yes,Yes,No,No,No,Yes,Yes,Yes
125,P99,head_neck_eyes_congenital_sensorineural_deafness,Head Neck Eyes,NaN,Head and neck,Sensorineural,Congenital Sensorineural hearing loss (Combine...,NaN,Congenital Sensorineural hearing loss (Combine...,Congenital Sensorineural hearing loss (Combine...,...,40,['H905'],Yes,Yes,No,No,No,Yes,Yes,Yes
126,P6,lymphatic_vascular_thrombotic_venous_events,Lymphatic Vascular,NaN,Vascular or lymphatic,Thrombosis,Thrombotic Events(Combined to Thrombosis),NaN,Thrombotic Events,Thrombotic Events,...,38,"['G458', 'G459', 'I248', 'I249', 'I258', 'G08'...",Yes,No,No,No,No,Yes,Yes,NaN
127,P109,lymphatic_vascular_venous_thrombosis,Lymphatic Vascular,NaN,Vascular or lymphatic,Thrombosis,Venous Thrombosis(Combined to Thrombosis),NaN,Venous Thrombosis,Venous Thrombosis,...,47,"['I82','I820','I821','I823','I828','I829']",Yes,No,No,No,No,Yes,Yes,NaN


In [ ]:
# Step 2: Load the ICD10 code descriptions file
icd10_df = pd.read_csv(r"coding19.tsv", sep='\t')
icd10_df


# Assume: 'coding' is the column with code, 'meaning' is the description
icd10_dict = dict(zip(icd10_df['coding'], icd10_df['meaning']))

,coding,meaning,node_id,parent_id,selectable
0,A00,A00 Cholera,286,23,N
1,A000,"A00.0 Cholera due to Vibrio cholerae 01, biova...",287,286,Y
2,A001,"A00.1 Cholera due to Vibrio cholerae 01, biova...",288,286,Y
3,A009,"A00.9 Cholera, unspecified",289,286,Y
4,A01,A01 Typhoid and paratyphoid fevers,290,23,N
...,...,...,...,...,...
19149,Z992,Z99.2 Dependence on renal dialysis,19150,19147,Y
19150,Z993,Z99.3 Dependence on wheelchair,19151,19147,Y
19151,Z994,Z99.4 Dependence on artificial heart,19152,19147,Y
19152,Z998,Z99.8 Dependence on other enabling machines an...,19153,19147,Y


In [316]:
gata2_phenotypes['ICD10_code'] = gata2_phenotypes['ICD10_code'].astype(str)

# Remove brackets and quotes, then split
gata2_phenotypes['ICD10_List'] = (
    gata2_phenotypes['ICD10_code']
    .str.replace(r"[\[\]']", "", regex=True)
    .str.split(',')
)

# Strip whitespace
gata2_phenotypes['ICD10_List'] = gata2_phenotypes['ICD10_List'].apply(lambda x: [code.strip() for code in x if code.strip() != ''])

# Explode to separate rows
exploded_df = gata2_phenotypes.explode('ICD10_List')
exploded_df['ICD10_List'] = exploded_df['ICD10_List'].str.strip()

In [317]:
# Map ICD10 code to meaning
exploded_df['ICD10_Description'] = exploded_df['ICD10_List'].map(icd10_dict)
exploded_df

,Phenotype_Index,Phenotype,Category,Category_Rank,New_Category,Order_Phenotype,Phenotype_Description_Expanded,Overlapping Phenotype Codes,Phenotype_Description,Phenotype_Description_v2,...,Present_in_GATA2,Present_in_UKB,Keep_NonZero,Keep_WithZero,Keep_Adults,Keep_OG,Keep_Everything,Keep_Combinations,ICD10_List,ICD10_Description
0,P100,head_neck_eyes_ptosis,Head Neck Eyes,1.0,Head and neck,1.01,Ptosis,NaN,Ptosis,Ptosis,...,Yes,Yes,Yes,Yes,Yes,Yes,Yes,Yes,H024,H02.4 Ptosis of eyelid
0,P100,head_neck_eyes_ptosis,Head Neck Eyes,1.0,Head and neck,1.01,Ptosis,NaN,Ptosis,Ptosis,...,Yes,Yes,Yes,Yes,Yes,Yes,Yes,Yes,Q100,Q10.0 Congenital ptosis
1,P102,head_neck_eyes_epicanthic_folds,Head Neck Eyes,1.0,Head and neck,1.02,Epicanthic folds,NaN,Epicanthic folds,Epicanthic folds,...,Yes,Yes,Yes,Yes,Yes,Yes,Yes,Yes,Q103,Q10.3 Other congenital malformations of eyelid
2,P101,head_neck_eyes_hypertelorism,Head Neck Eyes,1.0,Head and neck,1.03,Hypertelorism,NaN,Hypertelorism,Hypertelorism,...,Yes,Yes,Yes,Yes,Yes,Yes,Yes,Yes,Q752,Q75.2 Hypertelorism
3,P105,head_neck_eyes_strabismus,Head Neck Eyes,1.0,Head and neck,1.04,Strabismus,NaN,Strabismus,Strabismus,...,Yes,Yes,Yes,Yes,No,Yes,Yes,Yes,H49,H49 Paralytic strabismus
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
127,P109,lymphatic_vascular_venous_thrombosis,Lymphatic Vascular,NaN,Vascular or lymphatic,Thrombosis,Venous Thrombosis(Combined to Thrombosis),NaN,Venous Thrombosis,Venous Thrombosis,...,Yes,No,No,No,No,Yes,Yes,NaN,I821,I82.1 Thrombophlebitis migrans
127,P109,lymphatic_vascular_venous_thrombosis,Lymphatic Vascular,NaN,Vascular or lymphatic,Thrombosis,Venous Thrombosis(Combined to Thrombosis),NaN,Venous Thrombosis,Venous Thrombosis,...,Yes,No,No,No,No,Yes,Yes,NaN,I823,I82.3 Embolism and thrombosis of renal vein
127,P109,lymphatic_vascular_venous_thrombosis,Lymphatic Vascular,NaN,Vascular or lymphatic,Thrombosis,Venous Thrombosis(Combined to Thrombosis),NaN,Venous Thrombosis,Venous Thrombosis,...,Yes,No,No,No,No,Yes,Yes,NaN,I828,I82.8 Embolism and thrombosis of other specifi...
127,P109,lymphatic_vascular_venous_thrombosis,Lymphatic Vascular,NaN,Vascular or lymphatic,Thrombosis,Venous Thrombosis(Combined to Thrombosis),NaN,Venous Thrombosis,Venous Thrombosis,...,Yes,No,No,No,No,Yes,Yes,NaN,I829,I82.9 Embolism and thrombosis of unspecified vein


In [ ]:
exploded_df.to_csv(r"GATA2_Phenotypes_ICD10_Descriptions_Final_v8.csv", index=False)

In [318]:
icd10_df=exploded_df.copy()
icd10_df

,Phenotype_Index,Phenotype,Category,Category_Rank,New_Category,Order_Phenotype,Phenotype_Description_Expanded,Overlapping Phenotype Codes,Phenotype_Description,Phenotype_Description_v2,...,Present_in_GATA2,Present_in_UKB,Keep_NonZero,Keep_WithZero,Keep_Adults,Keep_OG,Keep_Everything,Keep_Combinations,ICD10_List,ICD10_Description
0,P100,head_neck_eyes_ptosis,Head Neck Eyes,1.0,Head and neck,1.01,Ptosis,NaN,Ptosis,Ptosis,...,Yes,Yes,Yes,Yes,Yes,Yes,Yes,Yes,H024,H02.4 Ptosis of eyelid
0,P100,head_neck_eyes_ptosis,Head Neck Eyes,1.0,Head and neck,1.01,Ptosis,NaN,Ptosis,Ptosis,...,Yes,Yes,Yes,Yes,Yes,Yes,Yes,Yes,Q100,Q10.0 Congenital ptosis
1,P102,head_neck_eyes_epicanthic_folds,Head Neck Eyes,1.0,Head and neck,1.02,Epicanthic folds,NaN,Epicanthic folds,Epicanthic folds,...,Yes,Yes,Yes,Yes,Yes,Yes,Yes,Yes,Q103,Q10.3 Other congenital malformations of eyelid
2,P101,head_neck_eyes_hypertelorism,Head Neck Eyes,1.0,Head and neck,1.03,Hypertelorism,NaN,Hypertelorism,Hypertelorism,...,Yes,Yes,Yes,Yes,Yes,Yes,Yes,Yes,Q752,Q75.2 Hypertelorism
3,P105,head_neck_eyes_strabismus,Head Neck Eyes,1.0,Head and neck,1.04,Strabismus,NaN,Strabismus,Strabismus,...,Yes,Yes,Yes,Yes,No,Yes,Yes,Yes,H49,H49 Paralytic strabismus
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
127,P109,lymphatic_vascular_venous_thrombosis,Lymphatic Vascular,NaN,Vascular or lymphatic,Thrombosis,Venous Thrombosis(Combined to Thrombosis),NaN,Venous Thrombosis,Venous Thrombosis,...,Yes,No,No,No,No,Yes,Yes,NaN,I821,I82.1 Thrombophlebitis migrans
127,P109,lymphatic_vascular_venous_thrombosis,Lymphatic Vascular,NaN,Vascular or lymphatic,Thrombosis,Venous Thrombosis(Combined to Thrombosis),NaN,Venous Thrombosis,Venous Thrombosis,...,Yes,No,No,No,No,Yes,Yes,NaN,I823,I82.3 Embolism and thrombosis of renal vein
127,P109,lymphatic_vascular_venous_thrombosis,Lymphatic Vascular,NaN,Vascular or lymphatic,Thrombosis,Venous Thrombosis(Combined to Thrombosis),NaN,Venous Thrombosis,Venous Thrombosis,...,Yes,No,No,No,No,Yes,Yes,NaN,I828,I82.8 Embolism and thrombosis of other specifi...
127,P109,lymphatic_vascular_venous_thrombosis,Lymphatic Vascular,NaN,Vascular or lymphatic,Thrombosis,Venous Thrombosis(Combined to Thrombosis),NaN,Venous Thrombosis,Venous Thrombosis,...,Yes,No,No,No,No,Yes,Yes,NaN,I829,I82.9 Embolism and thrombosis of unspecified vein


In [ ]:
#--------------------------Loading Dataset
#Load the ICD10 Diagnosis data 
data=pd.read_csv("DiagnosisICD10_41270.csv",sep =",",low_memory=False)
print(data.shape) 
data=data.set_index('eid')

(502175, 260)


In [ ]:
#Remove all the withdrawn participants from the dataset
withdrawn_pid=pd.read_csv(r"withdraw83200_480_20260109.txt",sep="\t",header=None)

In [322]:
data=data[~data.index.isin(withdrawn_pid[0])]
print("UKBiobank Control Dataset after removing withdrawn PID=",len(data.axes[0]))
#-----------------------------Clean Data
#Set PID as row index

data_df=data.dropna(inplace=False,how='all')
# ^ this line removes all the blank rows in UKBiobank database

print(len(data.axes[0])) # This command tells you the number of patients in the entire dataset
print(len(data_df.axes[0]))# Number of rows which has at least one value in one of the columns.
#N=440017/New number 446837


no_diagnosis=list(set(data.index)-set(data_df.index))
print("Number of rows/Participants with no ICD10 codes",len(no_diagnosis))


UKBiobank Control Dataset after removing withdrawn PID= 501932
501932
446632
Number of rows/Participants with no ICD10 codes 55300


Remove participants who withdrew

In [ ]:
variant_pid_list=pd.read_excel(r"UKB_Variants_PID_227Genes_Final.xlsx")

print("Number of Participants with P/LP variant",variant_pid_list['ParticipantID'].nunique(dropna=True))

Number of Participants with P/LP variant 55984


In [324]:
#Remove all rows with blank ParticipantID
variant_pid_list = variant_pid_list.dropna(subset=["ParticipantID"])

In [ ]:
#Read all the GATA2 participants in UKBiobank
gata2_variants_pid=pd.read_csv(r"GATA2_Variants_UKB.csv")

In [329]:
gata2_variants_pid['Source_File'].value_counts()

Source_File
3_128481046_C_T.csv      102
3_128481939_G_A.csv       84
3_128481926_C_T.csv        5
3_128481269_C_T.csv        2
3_128481901_G_T.csv        2
3_128481247_C_A.csv        1
3_128481301_G_GGT.csv      1
3_128483895_G_A.csv        1
Name: count, dtype: int64

In [330]:
variant_pid = list(
    set(
        variant_pid_list['ParticipantID'].dropna().tolist()
        + gata2_variants_pid['column'].dropna().tolist()
    )
)
print("Number of Participants with P/LP variant in 227 genes including GATA2",len(variant_pid))

Number of Participants with P/LP variant in 227 genes including GATA2 56157


Create a control for Combination analysis
Remove all the participants with P/LP variants in 227 genes

In [331]:
control_df=data_df.loc[~data_df.index.isin(variant_pid)]
print("Number of Control Participants without P/LP variant",len(control_df.axes[0]))
control_df

Number of Control Participants without P/LP variant 396360


,41270-0.0,41270-0.1,41270-0.2,41270-0.3,41270-0.4,41270-0.5,41270-0.6,41270-0.7,41270-0.8,41270-0.9,...,41270-0.249,41270-0.250,41270-0.251,41270-0.252,41270-0.253,41270-0.254,41270-0.255,41270-0.256,41270-0.257,41270-0.258
eid,,,,,,,,,,,,,,,,,,,,,
1000019,E039,E119,E780,I10,I209,I251,I259,K219,K297,K802,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1000022,M201,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1000035,O471,O701,O756,R739,Z349,Z360,Z370,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1000046,E039,F03,I10,N390,R031,R54,R55,Z740,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1000054,I839,L031,L539,M796,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6024068,B029,G439,K219,K30,M819,M858,N840,R11,Y415,Z871,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6024073,A099,B349,I10,I951,K229,K297,K318,K319,K573,K589,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6024095,K210,K449,K801,M4787,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


Subset GATA2 Cases

In [ ]:
# get ParticipantIDs for rows where Gene == 'GATA2'
gata2_pids = variant_pid_list.loc[variant_pid_list['Gene'] == 'GATA2', 'ParticipantID'].dropna()

# ensure IDs match the dtype of data.index (convert to int where possible)
try:
	gata2_pids = gata2_pids.astype(int).tolist()
except Exception:
	gata2_pids = [int(float(x)) for x in gata2_pids]

#gata2_pids

# 3. Combine, normalize dtype, and keep unique (order preserved)
# Combine with gata2_variants_pid['column'], drop NA, normalize dtype, keep unique (order preserved)
gata2_pids = (
    pd.Series(gata2_pids + gata2_variants_pid['column'].dropna().tolist())
      .astype(float)
      .astype(int)
      .unique()
      .tolist()
)

print("Number of GATA2 PLP Carriers:",len(gata2_pids))

# subset data by those participant IDs (eids)
gata2_cases = data_df.loc[data_df.index.isin(gata2_pids)]
print("Number of GATA2 cases with non missing ICD10 dataset:", len(gata2_cases.axes[0]))
gata2_cases.to_csv("GATA2_177Cases_ICD10.csv")

Number of GATA2 PLP Carriers: 198
Number of GATA2 cases with non missing ICD10 dataset: 177


Remove participants who have any cancer

In [ ]:
#Read all the participants who do not have any the cancer ICD10 code in p40006,p41270,p41271
noncancer_ukb=pd.read_csv(r"Negative_Cancer_Phenotype_ICD10_ICD9_Cancer.csv")
noncancer_ukb

,eid,Sex,Age_at_recruitment,ICD10,N_Cancer,Cancer_Date,Collapsed_p40006,Age_at_Cancer,Histology,Behavior,...,p41270,p41270_list,has_icd10_match,matching_icd10,matching_icd10_meaning,p41271_list,matching_icd9,has_icd9_match,matching_icd9_meaning,_merge
0,1000019,Male,63.0,"['E039', 'E119', 'E780', 'I10', 'I209', 'I251'...",NaN,NaN,NaN,NaN,NaN,NaN,...,"['E039', 'E119', 'E780', 'I10', 'I209', 'I251'...","['E039', 'E119', 'E780', 'I10', 'I209', 'I251'...",False,[],[],[],[],False,[],both
1,1000141,Female,58.0,"['I10', 'J459', 'N811', 'Z880', 'Z910']",NaN,NaN,NaN,NaN,NaN,NaN,...,"['I10', 'J459', 'N811', 'Z880', 'Z910']","['I10', 'J459', 'N811', 'Z880', 'Z910']",False,[],[],[],[],False,[],both
2,1000186,Male,43.0,['K409'],NaN,NaN,NaN,NaN,NaN,NaN,...,['K409'],['K409'],False,[],[],[],[],False,[],both
3,1000255,Male,49.0,"['K449', 'N359', 'N40', 'Z115', 'Z888']",NaN,NaN,NaN,NaN,NaN,NaN,...,"['K449', 'N359', 'N40', 'Z115', 'Z888']","['K449', 'N359', 'N40', 'Z115', 'Z888']",False,[],[],[],[],False,[],both
4,1000304,Female,63.0,"['I10', 'S3200', 'V798']",NaN,NaN,NaN,NaN,NaN,NaN,...,"['I10', 'S3200', 'V798']","['I10', 'S3200', 'V798']",False,[],[],[],[],False,[],both
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
374268,6023947,Female,58.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,[],False,[],[],[],[],False,[],both
374269,6023959,Male,46.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,[],False,[],[],[],[],False,[],both
374270,6024030,Male,65.0,"['D176', 'E119', 'E559', 'E780', 'F171', 'F329...",NaN,NaN,NaN,NaN,NaN,NaN,...,"['D176', 'E119', 'E559', 'E780', 'F171', 'F329...","['D176', 'E119', 'E559', 'E780', 'F171', 'F329...",False,[],[],[],[],False,[],both
374271,6024068,Female,58.0,"['B029', 'G439', 'K219', 'K30', 'M819', 'M858'...",NaN,NaN,NaN,NaN,NaN,NaN,...,"['B029', 'G439', 'K219', 'K30', 'M819', 'M858'...","['B029', 'G439', 'K219', 'K30', 'M819', 'M858'...",False,[],[],[],[],False,[],both


In [334]:
control_df=control_df[control_df.index.isin(noncancer_ukb['eid'])]
print("UKBiobank Control Dataset=",len(control_df.axes[0]))

UKBiobank Control Dataset= 287034


In [335]:
# Select columns correctly using a list
data_df_merge = control_df.merge(
	noncancer_ukb[['eid', 'Sex', 'Age_at_recruitment']],
	how='left',
	on='eid'
)

In [ ]:
print(data_df_merge['Sex'].value_counts())



Sex
Female    158555
Male      128479
Name: count, dtype: int64
count    287034.000000
mean         55.872754
std           8.132119
min          37.000000
25%          49.000000
50%          57.000000
75%          63.000000
max          73.000000
Name: Age_at_recruitment, dtype: float64


In [337]:
# Select columns correctly using a list
gata2_df_merge = gata2_cases.merge(
	noncancer_ukb[['eid', 'Sex', 'Age_at_recruitment']],
	how='left',
	on='eid'
)

len(gata2_cases)


177

In [338]:
print(gata2_df_merge['Sex'].value_counts())

Sex
Female    77
Male      59
Name: count, dtype: int64


In [ ]:
import pandas as pd

# Assume data_df looks like this:
# Columns: eid, diag_1, diag_2, diag_3, ...
# Example ICD10 codes are present in diag_* columns

# STEP 1: Define columns to search (all columns except 'eid')
icd10_cols = [col for col in control_df.columns if col != 'eid']

# STEP 2: Flatten the unique ICD10 codes from phenotype_df
all_icd10_codes = set(icd10_df['ICD10_List'].dropna().unique())

# STEP 3: For each ICD10 code, find eids
icd10_to_eids = {}

for icd10 in all_icd10_codes:
    # Check for presence in any diag_* column
    mask = control_df[icd10_cols].isin([icd10]).any(axis=1)
    eids = control_df.index[mask].tolist()
    icd10_to_eids[icd10] = eids

# OPTIONAL: save as dictionary or dataframe
# Example: convert to DataFrame
icd10_eid_df = pd.DataFrame([
    {'ICD10': code, 'EIDs': eids, 'N': len(eids)}
    for code, eids in icd10_to_eids.items()
])





In [ ]:
# Find max list length for padding
max_len = max(len(eids) for eids in icd10_to_eids.values())

# Pad lists with NaN and create DataFrame
icd10_eid_df = pd.DataFrame({
    code: eids + [pd.NA] * (max_len - len(eids))
    for code, eids in icd10_to_eids.items()
})




          Q279  C927  C863    M5409     Q646  C845   J44     A600     A488  \
0      1051252  <NA>  <NA>  1783273  4765460  <NA>  <NA>  1155819  1107497   
1      1160802  <NA>  <NA>  2526369     <NA>  <NA>  <NA>  1325816  1423317   
2      1286004  <NA>  <NA>  5058514     <NA>  <NA>  <NA>  1453710  1827915   
3      1540841  <NA>  <NA>  5090971     <NA>  <NA>  <NA>  1455212  2973253   
4      1759246  <NA>  <NA>  5757961     <NA>  <NA>  <NA>  2164769  3777094   
...        ...   ...   ...      ...      ...   ...   ...      ...      ...   
30709     <NA>  <NA>  <NA>     <NA>     <NA>  <NA>  <NA>     <NA>     <NA>   
30710     <NA>  <NA>  <NA>     <NA>     <NA>  <NA>  <NA>     <NA>     <NA>   
30711     <NA>  <NA>  <NA>     <NA>     <NA>  <NA>  <NA>     <NA>     <NA>   
30712     <NA>  <NA>  <NA>     <NA>     <NA>  <NA>  <NA>     <NA>     <NA>   
30713     <NA>  <NA>  <NA>     <NA>     <NA>  <NA>  <NA>     <NA>     <NA>   

          Z855  ...     I820      B09      C97     Q639      C5

In [ ]:
# 'coding' is the column with code, 'meaning' is the description
icd10_dict = dict(zip(icd10_df['coding'], icd10_df['meaning']))


In [ ]:
icd10_eid_df_labels=icd10_eid_df.copy()
# After icd10_eid_df has been created (columns are ICD10 codes), map meanings into column names
def make_col_label(code):
    meaning = icd10_dict.get(code)
    if meaning is None or (isinstance(meaning, float) and pd.isna(meaning)):
        return str(code)
    # short-clean the meaning to avoid very long names if desired; here keep full meaning
    return f"{meaning}"

# apply mapping
new_columns = [make_col_label(col) for col in icd10_eid_df.columns]

icd10_eid_df_labels.columns = new_columns
icd10_eid_df_labels
# Optional: if you need to also keep the original codes, create a mapping DataFrame
# colmap_df = pd.DataFrame({'code': icd10_eid_df.columns, 'label': new_columns})

# Save to CSV/Excel with new descriptive headers
icd10_eid_df.to_csv("ICD10_EID_Match_Columns_with_meaning_030226.csv", index=False)


In [ ]:
import pandas as pd

phenotype_df = pd.read_excel("GATA2_Phenotypes_Final_v8.xlsx")
phenotype_df

,Phenotype_Index,Phenotype,Category,Category_Rank,New_Category,Order_Phenotype,Phenotype_Description_Expanded,Overlapping Phenotype Codes,Phenotype_Description,Phenotype_Description_v2,...,Rank,ICD10_code,Present_in_GATA2,Present_in_UKB,Keep_NonZero,Keep_WithZero,Keep_Adults,Keep_OG,Keep_Everything,Keep_Combinations
0,P100,head_neck_eyes_ptosis,Head Neck Eyes,1.0,Head and neck,1.01,Ptosis,NaN,Ptosis,Ptosis,...,49,"['H024','Q100']",Yes,Yes,Yes,Yes,Yes,Yes,Yes,Yes
1,P102,head_neck_eyes_epicanthic_folds,Head Neck Eyes,1.0,Head and neck,1.02,Epicanthic folds,NaN,Epicanthic folds,Epicanthic folds,...,51,['Q103'],Yes,Yes,Yes,Yes,Yes,Yes,Yes,Yes
2,P101,head_neck_eyes_hypertelorism,Head Neck Eyes,1.0,Head and neck,1.03,Hypertelorism,NaN,Hypertelorism,Hypertelorism,...,53,['Q752'],Yes,Yes,Yes,Yes,Yes,Yes,Yes,Yes
3,P105,head_neck_eyes_strabismus,Head Neck Eyes,1.0,Head and neck,1.04,Strabismus,NaN,Strabismus,Strabismus,...,55,"['H49','H500', 'H509', 'H501', 'H502', 'H506',...",Yes,Yes,Yes,Yes,No,Yes,Yes,Yes
4,P104,head_neck_eyes_small_palpebral_fissures,Head Neck Eyes,1.0,Head and neck,1.05,Short palpebral fissures,NaN,Short palpebral fissures,Short palpebral fissures,...,54,['H025'],Yes,Yes,Yes,Yes,Yes,Yes,Yes,Yes
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
124,P12,head_neck_eyes_treatment_associated_hearing_loss,Head Neck Eyes,NaN,Head and neck,Sensorineural,Treatment-associated hearing loss (Combined to...,NaN,Treatment-associated hearing loss (Combined to...,Treatment-associated hearing loss (Combined to...,...,38,"['H900', 'H901', 'H902', 'H903', 'H904', 'H906...",Yes,Yes,No,No,No,Yes,Yes,Yes
125,P99,head_neck_eyes_congenital_sensorineural_deafness,Head Neck Eyes,NaN,Head and neck,Sensorineural,Congenital Sensorineural hearing loss (Combine...,NaN,Congenital Sensorineural hearing loss (Combine...,Congenital Sensorineural hearing loss (Combine...,...,40,['H905'],Yes,Yes,No,No,No,Yes,Yes,Yes
126,P6,lymphatic_vascular_thrombotic_venous_events,Lymphatic Vascular,NaN,Vascular or lymphatic,Thrombosis,Thrombotic Events(Combined to Thrombosis),NaN,Thrombotic Events,Thrombotic Events,...,38,"['G458', 'G459', 'I248', 'I249', 'I258', 'G08'...",Yes,No,No,No,No,Yes,Yes,NaN
127,P109,lymphatic_vascular_venous_thrombosis,Lymphatic Vascular,NaN,Vascular or lymphatic,Thrombosis,Venous Thrombosis(Combined to Thrombosis),NaN,Venous Thrombosis,Venous Thrombosis,...,47,"['I82','I820','I821','I823','I828','I829']",Yes,No,No,No,No,Yes,Yes,NaN


In [ ]:

# STEP 1: Create a dictionary to hold lists of eid for each Phenotype_Index
phenotype_index_to_eids = {}

for idx, row in phenotype_df.iterrows():
    phenotype_index = row['Phenotype_Index']
    
    # Clean ICD10 code list for this phenotype
    icd10_codes = row['ICD10_code']
    if isinstance(icd10_codes, str):
        icd10_codes = icd10_codes.replace("[", "").replace("]", "").replace("'", "").split(",")
        icd10_codes = [code.strip() for code in icd10_codes if code.strip() != '']
    else:
        icd10_codes = []
    
    # Collect all eids who have at least one of these ICD10 codes
    eid_set = set()
    for code in icd10_codes:
        if code in icd10_eid_df.columns:
            eids = icd10_eid_df[code].dropna().tolist()
            eid_set.update(eids)
    
    phenotype_index_to_eids[phenotype_index] = list(eid_set)

# STEP 2: Determine max length for padding
max_len = max(len(eids) for eids in phenotype_index_to_eids.values())

# Create DataFrame with columns = Phenotype_Index, values = eid lists (padded with NA)
phenotype_eid_wide_df = pd.DataFrame({
    phenotype_index: eids + [pd.NA] * (max_len - len(eids))
    for phenotype_index, eids in phenotype_index_to_eids.items()
})






          P100     P102  P101     P105     P104     P106       P9       P8  \
0      5439488  4722209  <NA>  2531330  5158400  4740610  3881482  1458688   
1      4505601  3570565  <NA>  1540100  5293572  3436038  2602637  2605569   
2      1236999  2650822  <NA>  3907588  3299333  3254537  3591312  3831297   
3      2613267  6000103  <NA>  1005575  3756552  2576524  4137878  2591236   
4      5791767  5262124  <NA>  4954119  3470862  1857677  2062110  3494405   
...        ...      ...   ...      ...      ...      ...      ...      ...   
31281     <NA>     <NA>  <NA>     <NA>     <NA>     <NA>     <NA>     <NA>   
31282     <NA>     <NA>  <NA>     <NA>     <NA>     <NA>     <NA>     <NA>   
31283     <NA>     <NA>  <NA>     <NA>     <NA>     <NA>     <NA>     <NA>   
31284     <NA>     <NA>  <NA>     <NA>     <NA>     <NA>     <NA>     <NA>   
31285     <NA>     <NA>  <NA>     <NA>     <NA>     <NA>     <NA>     <NA>   

           P11     P130  ...     P108  P125   P48      P77   P6

In [ ]:
import pandas as pd

# Read the EIDs from the text file
with open("participant_id_low_monocytes.txt", "r") as file:
    low_mono_eids = [line.strip() for line in file if line.strip() != '']

# Ensure they are unique (optional)
low_mono_eids = list(set(low_mono_eids))

# Replace/add them in P37 column:
max_len = max(len(low_mono_eids), len(phenotype_eid_wide_df))

# Ensure phenotype_eid_wide_df can accommodate the new length
if len(phenotype_eid_wide_df) < max_len:
    # Pad existing DataFrame with rows
    additional_rows = max_len - len(phenotype_eid_wide_df)
    padding_df = pd.DataFrame([ [pd.NA] * len(phenotype_eid_wide_df.columns) ] * additional_rows, columns=phenotype_eid_wide_df.columns)
    phenotype_eid_wide_df = pd.concat([phenotype_eid_wide_df, padding_df], ignore_index=True)

# Now replace the P37 column
phenotype_eid_wide_df['P37'] = low_mono_eids + [pd.NA] * (max_len - len(low_mono_eids))

# View result
print(phenotype_eid_wide_df)

# OPTIONAL: Save updated DataFrame
phenotype_eid_wide_df.to_excel("Phenotype_EID_Wide_v8_030226.xlsx", index=False)


          P100     P102  P101     P105     P104     P106       P9       P8  \
0      5439488  4722209  <NA>  2531330  5158400  4740610  3881482  1458688   
1      4505601  3570565  <NA>  1540100  5293572  3436038  2602637  2605569   
2      1236999  2650822  <NA>  3907588  3299333  3254537  3591312  3831297   
3      2613267  6000103  <NA>  1005575  3756552  2576524  4137878  2591236   
4      5791767  5262124  <NA>  4954119  3470862  1857677  2062110  3494405   
...        ...      ...   ...      ...      ...      ...      ...      ...   
31281     <NA>     <NA>  <NA>     <NA>     <NA>     <NA>     <NA>     <NA>   
31282     <NA>     <NA>  <NA>     <NA>     <NA>     <NA>     <NA>     <NA>   
31283     <NA>     <NA>  <NA>     <NA>     <NA>     <NA>     <NA>     <NA>   
31284     <NA>     <NA>  <NA>     <NA>     <NA>     <NA>     <NA>     <NA>   
31285     <NA>     <NA>  <NA>     <NA>     <NA>     <NA>     <NA>     <NA>   

           P11     P130  ...     P108  P125   P48      P77   P6

In [292]:
# 1. Remove withdrawn participants
withdrawn_set = set(withdrawn_pid[0].astype(str).str.strip().values)
low_mono_eids_trim = [
    eid.strip() for eid in low_mono_eids 
    if str(eid).strip() not in withdrawn_set
]

# 2. Remove cancer-risk eids (pid_cancer_risk_all)
if 'eid' in variant_pid_list.columns:
    cancer_risk_set = set(variant_pid_list['ParticipantID'].astype(str).str.strip().values)
else:
    cancer_risk_set = set(variant_pid_list.iloc[:, 0].astype(str).str.strip().values)

low_mono_eids_trim = [
    eid for eid in low_mono_eids_trim 
    if str(eid).strip() not in cancer_risk_set
]

# 3. Remove eids present in no_diagnosis
no_diag_set = {str(x).strip() for x in no_diagnosis}

low_mono_eids_trim = [
    eid for eid in low_mono_eids_trim 
    if str(eid).strip() not in no_diag_set
]

print("Number of low monocyte eids after trimming:", len(low_mono_eids_trim))

Number of low monocyte eids after trimming: 4305


In [ ]:
import pandas as pd

gata2_pid_clean = (
    pd.Series(gata2_pids)
      .astype(str)
      .str.strip()
      .pipe(pd.to_numeric, errors="coerce")
      .dropna()
      .astype(int)
      .tolist()
)

low_mono_eids_trim_clean = (
    pd.Series(low_mono_eids_trim)
      .astype(str)
      .str.strip()
      .pipe(pd.to_numeric, errors="coerce")
      .dropna()
      .astype(int)
      .tolist()
)

# overlap check
overlap = list(set(gata2_pid_clean) & set(low_mono_eids_trim_clean))



([3031888, 1713508, 4732492, 1849294], 4)

In [295]:
low_mono_eids_trim_clean = list(
    set(low_mono_eids_trim_clean) - set(gata2_pid_clean)
)


In [ ]:
# ...existing code...
# Ensure consistent string types and trim whitespace
eid_set = set(str(x).strip() for x in low_mono_eids_trim_clean)

selected = [['eid', 'Sex', 'Age_at_recruitment']].copy()
selected = selected[selected['eid'].astype(str).str.strip().isin(eid_set)]

# View or save
print(selected.shape)

# ...existing code...

(4301, 3)


,eid,Sex,Age_at_recruitment
54,1002989,Female,48.0
85,1004913,Male,61.0
255,1013450,Male,57.0
437,1022459,Female,69.0
545,1028141,Male,55.0
...,...,...,...
501840,5995796,Female,43.0
501875,5997255,Female,66.0
502112,6009461,Female,68.0
502216,6014681,Female,46.0


In [ ]:
# ...existing code...
# Create data_merge_final from data_df_merge then add selected and remove duplicate eids
data_merge_final = data_df_merge[['eid', 'Sex', 'Age_at_recruitment']].copy()

# Normalize eid strings to avoid mismatches
data_merge_final['eid'] = data_merge_final['eid'].astype(str).str.strip()
selected['eid'] = selected['eid'].astype(str).str.strip()

# Append selected and drop duplicated eids (keep first occurrence)
data_merge_final = pd.concat([data_merge_final, selected], ignore_index=True)
data_merge_final = data_merge_final.drop_duplicates(subset='eid', keep='first').reset_index(drop=True)

# Inspect / save
print(data_merge_final.shape)
data_merge_final.head()
# Optionally save:
data_merge_final.to_csv("Control_GATA2_EID_Final_030226.csv", index=False)
# ...existing code...

(288600, 3)


In [ ]:
# Ensure all columns are padded to max rows (optional but safe step)
max_len = max(phenotype_eid_wide_df[col].notna().sum() for col in phenotype_eid_wide_df.columns)

if len(phenotype_eid_wide_df) < max_len:
    additional_rows = max_len - len(phenotype_eid_wide_df)
    padding_df = pd.DataFrame(
        [[pd.NA] * len(phenotype_eid_wide_df.columns)] * additional_rows,
        columns=phenotype_eid_wide_df.columns
    )
    phenotype_eid_wide_df = pd.concat([phenotype_eid_wide_df, padding_df], ignore_index=True)

# Fill blanks (NaN) with 0 (as integer or string depending on your need)
phenotype_eid_wide_df = phenotype_eid_wide_df.fillna(0)

# OPTIONAL: Convert to integer type if desired (only works if all values are numeric after fillna)
# phenotype_eid_wide_df = phenotype_eid_wide_df.astype(int)

# Save to Excel
phenotype_eid_wide_df.to_excel("x   Phenotype_pid_final_022526_v8.xlsx", index=False)
